# Prompt Garden Control Notebook

This notebook is the authoring and registration surface for Prompt Garden.

Use it to:

- register prompt roots, descendants, and few-shot sets
- generate combos and choose which ones belong to an experiment
- create experiments and prepare execution commands

Do not use it as the main answer-review surface.
The intended long-term workflow is:

1. notebook for authoring and experiment setup
2. script runner for execution
3. Streamlit app for comparison and review

The dedicated runner now exists as `scripts/run_prompt_experiment.py`.
A temporary inline execution fallback still remains below as a legacy bridge.


In [ ]:
from pathlib import Path
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'chemistry_bot').exists():
            return candidate
    raise RuntimeError('Could not find project root containing src/chemistry_bot.')

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / 'src'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.append(str(SRC_ROOT))

from chemistry_bot.promptops.garden import PromptGarden

garden = PromptGarden(PROJECT_ROOT / 'prompt_garden')
garden.init()

baseline_manifest_path = PROJECT_ROOT / 'prompt_garden' / 'curated' / 'baseline_manifest.json'
baseline_manifest = json.loads(baseline_manifest_path.read_text(encoding='utf-8'))

print('PromptGarden')
print('root:', garden.root)
print('recommended baseline combo:', baseline_manifest['combo_id'])
print('recommended baseline fewshot:', baseline_manifest['fewshot_id'])
garden.print_summary()


## 1. Seed baseline prompts only if the registry is empty

This cell is safe to keep in the notebook, but it only creates starter prompts when no prompt nodes exist yet.


In [ ]:
SYSTEM_PROMPT_V1 = """
You are a chemistry teacher in a {type_of_school}.
Answer in {language}.

Main rules:
1. Factual accuracy is more important than a beautiful explanation.
2. Do not invent colors, odors, aggregate states, reaction conditions,
   reagent concentrations, amounts, or experimental procedures.
3. Correct false assumptions in the student's question.
4. If reliable information is insufficient, say so clearly.
5. Follow the Pydantic response schema. Do not add text outside it.
"""

USER_PROMPT_V1 = """
Process the following chemistry request.

Student class: {school_class}
Explanation level: {explanation_level}
Question: {question}

Intent rules:
1. First determine request_type: theory, experiment, or mixed.
2. Follow the user's literal intent.
3. Never interpret "experiment with a substance" as
   "synthesize or manufacture the substance" unless the user explicitly asks.
4. If the requested experimental goal is ambiguous,
   return experiment.kind="clarification".
5. Never invent amounts, concentrations, temperatures, risks or disposal instructions.

Verified protocol context:
{protocol_context}
"""

if not garden.list_nodes():
    sys_node = garden.create_root(
        prompt_type='system',
        tree_id='system_chemistry_main',
        title='Base chemistry system prompt',
        text=SYSTEM_PROMPT_V1,
        tags=['chemistry', 'system', 'safety'],
    )

    usr_node = garden.create_root(
        prompt_type='user',
        tree_id='user_chemistry_main',
        title='Base chemistry user prompt',
        text=USER_PROMPT_V1,
        tags=['chemistry', 'user', 'intent'],
    )

    combo = garden.create_combo(
        title='Chemistry CLI baseline combo',
        prompt_ids={
            'system': sys_node['id'],
            'user': usr_node['id'],
        },
        status='untested',
        test_status='untested',
        notes='Initial system + user prompt combo.',
        tags=['chemistry', 'baseline'],
    )

    print('Created:', sys_node['id'], usr_node['id'], combo['id'])
else:
    print('Registry already has prompt nodes.')

garden.print_summary()


## 1.1 Register few-shot baselines idempotently

These cells validate the local few-shot files and only register the root or child node if the expected ID is missing.


In [ ]:
fewshot_root_path = PROJECT_ROOT / 'prompt_garden' / 'prompts' / 'fewshot' / 'fsh_000001.md'
assert fewshot_root_path.exists(), f'Missing local few-shot file: {fewshot_root_path}'

FEW_SHOT_CORE_V1 = json.loads(fewshot_root_path.read_text(encoding='utf-8'))
assert isinstance(FEW_SHOT_CORE_V1, list)
assert len(FEW_SHOT_CORE_V1) == 5

for example in FEW_SHOT_CORE_V1:
    assert 'id' in example
    assert 'input' in example
    assert 'answer' in example

try:
    fewshot_root = garden.get_node('fsh_000001')
    print('Few-shot root already registered:', fewshot_root['id'])
except KeyError:
    fewshot_root = garden.create_root(
        prompt_type='fewshot',
        tree_id='fewshot_chemistry_main',
        title='Core chemistry golden few-shot v1',
        text=json.dumps(FEW_SHOT_CORE_V1, ensure_ascii=False, indent=2),
        tags=['chemistry', 'fewshot', 'golden', 'static'],
        metadata={
            'kind': 'fewshot_set',
            'format': 'json',
            'selection': 'static',
            'example_count': len(FEW_SHOT_CORE_V1),
        },
    )

fewshot_root


In [ ]:
fewshot_child_path = PROJECT_ROOT / 'prompt_garden' / 'prompts' / 'fewshot' / 'fsh_000002.md'
assert fewshot_child_path.exists(), f'Missing local few-shot file: {fewshot_child_path}'

FEW_SHOT_CORE_V2 = json.loads(fewshot_child_path.read_text(encoding='utf-8'))

try:
    fewshot_child = garden.get_node('fsh_000002')
    print('Few-shot child already registered:', fewshot_child['id'])
except KeyError:
    fewshot_child = garden.create_child(
        parent_id='fsh_000001',
        title='English rules few-shot v2',
        text=json.dumps(FEW_SHOT_CORE_V2, ensure_ascii=False, indent=2),
        tags=['chemistry', 'fewshot', 'english', 'safety'],
        branch='safety',
        metadata={
            'kind': 'fewshot_set',
            'format': 'json',
            'selection': 'static',
            'example_count': len(FEW_SHOT_CORE_V2),
        },
    )

fewshot_child


## 2. Create prompt descendants when you want a new branch

Keep authoring here. Do not put large-scale result review back into the notebook.


In [ ]:
# Example: create a short system descendant.
# Uncomment and edit when you want a new branch.

# short_system = """
# You are a strict chemistry teacher.
# Answer in {language}.
# Be concise, accurate, and safe.
# Do not invent experimental details.
# Follow the Pydantic schema.
# """
#
# garden.create_child(
#     parent_id=baseline_manifest['prompt_ids']['system'],
#     title='Short strict system prompt',
#     text=short_system,
#     tags=['chemistry', 'short', 'strict'],
#     branch='short',
# )

# Example: create a stricter user descendant.
# strict_user = garden.read_prompt(baseline_manifest['prompt_ids']['user']) + """
#
# Additional safety rule:
# If the student asks about oxidizers, fuel mixtures, heating, or unknown concentrations,
# prefer experiment.kind='none' or experiment.kind='clarification'.
# """
#
# garden.create_child(
#     parent_id=baseline_manifest['prompt_ids']['user'],
#     title='User prompt with stronger oxidizer safety rule',
#     text=strict_user,
#     tags=['chemistry', 'oxidizer', 'safety'],
#     branch='safety',
# )


In [ ]:
# Example: create a new few-shot descendant.
# Edit the source file or inline JSON first, then register the child node.

# next_fewshot_path = PROJECT_ROOT / 'prompt_garden' / 'prompts' / 'fewshot' / 'fsh_000003.md'
# next_fewshot_examples = json.loads(next_fewshot_path.read_text(encoding='utf-8'))
#
# garden.create_child(
#     parent_id=baseline_manifest['fewshot_id'],
#     title='Few-shot v3 candidate',
#     text=json.dumps(next_fewshot_examples, ensure_ascii=False, indent=2),
#     tags=['chemistry', 'fewshot', 'candidate'],
#     branch='candidate',
#     metadata={
#         'kind': 'fewshot_set',
#         'format': 'json',
#         'selection': 'static',
#         'example_count': len(next_fewshot_examples),
#     },
# )


## 3. Generate candidate combos

Re-running this cell is safe because combo generation uses a stable combo key and skips existing combinations.


In [ ]:
created = garden.generate_combos(
    roles_to_prompt_type={
        'system': 'system',
        'user': 'user',
    },
    title_prefix='Chemistry generated combo',
    include_context_variants=True,
    skip_existing=True,
)

print(f'Created {len(created)} new combos')
garden.print_summary()


## 4. Inspect untested combos and choose which ones to attach

Use `ONLY_COMBO_IDS` for a narrow run or `SKIP_COMBO_IDS` when you want to exclude selected candidates from a larger batch.


In [ ]:
untested = garden.list_untested_combos()
print('Untested combo count:', len(untested))
print('Recommended baseline combo:', baseline_manifest['combo_id'])
untested[:10]


In [ ]:
ONLY_COMBO_IDS = []
SKIP_COMBO_IDS = []

candidate_combos = garden.list_untested_combos()

if ONLY_COMBO_IDS:
    selected_combos = [
        combo for combo in candidate_combos
        if combo['id'] in set(ONLY_COMBO_IDS)
    ]
else:
    selected_combos = [
        combo for combo in candidate_combos
        if combo['id'] not in set(SKIP_COMBO_IDS)
    ]

selected_combo_ids = [combo['id'] for combo in selected_combos]

{
    'candidate_count': len(candidate_combos),
    'selected_count': len(selected_combo_ids),
    'selected_combo_ids_preview': selected_combo_ids[:10],
    'skipped_combo_ids': SKIP_COMBO_IDS,
}


## 5. Register or load an experiment

Keep the experiment spec explicit. Reuse an existing experiment by setting `EXISTING_EXPERIMENT_ID`, or create a new one by flipping `CREATE_EXPERIMENT` to `True`.


In [ ]:
EXPERIMENT_SPEC = {
    'name': 'short_vs_context_prompt_smoke_test_few_shot_small_model',
    'goal': 'Compare prompt variants on school-level chemistry safety and theory cases.',
    'hypothesis': 'The English curated few-shot set improves consistency on the default chemistry cases.',
    'notes': 'Notebook prepares the experiment; execution should move to the script runner.',
    'tags': ['chemistry', 'combo', 'smoke-test', 'safety'],
}

EXPERIMENT_SPEC


In [ ]:
EXISTING_EXPERIMENT_ID = None
CREATE_EXPERIMENT = False

if EXISTING_EXPERIMENT_ID:
    experiment = garden.get_experiment(EXISTING_EXPERIMENT_ID)
    print('Loaded existing experiment:', experiment['id'])
elif CREATE_EXPERIMENT:
    experiment = garden.create_experiment(**EXPERIMENT_SPEC)
    print('Created new experiment:', experiment['id'])
else:
    print('Set EXISTING_EXPERIMENT_ID or flip CREATE_EXPERIMENT = True when you are ready.')


## 6. Attach the selected combos and review experiment membership


In [ ]:
if 'experiment' not in globals():
    raise RuntimeError('Load or create an experiment first.')

EXPERIMENT_ID = experiment['id']

garden.attach_combos_to_experiment(
    experiment_id=EXPERIMENT_ID,
    combo_ids=selected_combo_ids,
    notes='Prepared from the control notebook before scripted execution.',
)

experiment_state = garden.get_experiment(EXPERIMENT_ID)
{
    'experiment_id': experiment_state['id'],
    'status': experiment_state['status'],
    'attached_combo_count': len(experiment_state.get('combo_ids', [])),
    'attached_combo_preview': experiment_state.get('combo_ids', [])[:10],
    'untested_attached_combo_count': len(garden.list_experiment_untested_combos(EXPERIMENT_ID)),
}


## 7. Generate the planned runner and review commands

The notebook should stop at experiment preparation. The commands below are the handoff point to the real script runner and the future review app.


In [ ]:
if 'experiment' not in globals():
    raise RuntimeError('Load or create an experiment first.')

MODEL_NAME = 'phi4-mini:latest'
FEWSHOT_ID = baseline_manifest['fewshot_id']
PYTHON_CMD = './.venv/Scripts/python.exe' if (PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe').exists() else 'python'
RUNNER_SCRIPT = PROJECT_ROOT / 'scripts' / 'run_prompt_experiment.py'
REVIEW_APP = PROJECT_ROOT / 'apps' / 'prompt_garden_review.py'

runner_command = (
    f'{PYTHON_CMD} scripts/run_prompt_experiment.py '
    f'--experiment-id {EXPERIMENT_ID} '
    f'--model "{MODEL_NAME}" '
    f'--fewshot-id {FEWSHOT_ID}'
)

selected_runner_command = runner_command
if selected_combo_ids:
    selected_runner_command += ' ' + ' '.join(
        f'--only-combo {combo_id}'
        for combo_id in selected_combo_ids
    )

review_command = (
    f'streamlit run apps/prompt_garden_review.py '
    f'-- --experiment-id {EXPERIMENT_ID}'
)

{
    'runner_script_exists_today': RUNNER_SCRIPT.exists(),
    'review_app_exists_today': REVIEW_APP.exists(),
    'runner_command_full_batch': runner_command,
    'runner_command_selected_combos': selected_runner_command,
    'planned_review_command': review_command,
}


## 8. Temporary fallback: inline execution inside the notebook

This block remains only as a bridge until `scripts/run_prompt_experiment.py` exists.
Prefer the command handoff above now that the dedicated runner exists.


In [ ]:
from chemistry_bot.cli.legacy_bot import CliBot
from chemistry_bot.promptops.eval import (
    DEFAULT_CHEMISTRY_CASES,
    compact_rows,
    evaluate_case,
    summarize_results,
)

if 'experiment' not in globals():
    raise RuntimeError('Load or create an experiment first.')

combos_to_test = [garden.get_combo(combo_id) for combo_id in selected_combo_ids]
if not combos_to_test:
    combos_to_test = garden.list_experiment_untested_combos(EXPERIMENT_ID)

print('Combos to test:', [combo['id'] for combo in combos_to_test])

all_result_rows = []

for combo in combos_to_test:
    combo_id = combo['id']
    print('\nTesting', combo_id)

    bot = CliBot(
        model_name=MODEL_NAME,
        garden_root=garden.root,
        combo_id=combo_id,
        fewshot_id=FEWSHOT_ID,
        max_history_messages=12,
        materialize_context=True,
    )

    case_results = []

    for case in DEFAULT_CHEMISTRY_CASES:
        bot.student_context.protocol_context = case.get(
            'protocol_context',
            'No verified experimental protocol was retrieved.',
        )

        answer = bot.invoke_once(
            user_text=case['question'],
            session_id=f'{EXPERIMENT_ID}_{combo_id}_{case["id"]}',
            experiment_id=EXPERIMENT_ID,
            silent=True,
        )

        case_results.append(evaluate_case(answer, case))

    summary = summarize_results(case_results)

    garden.record_experiment_combo_result(
        experiment_id=EXPERIMENT_ID,
        combo_id=combo_id,
        score=summary['average_score'],
        result_text=f'Auto smoke test completed. pass_rate={summary["pass_rate"]}',
        subject_score=None,
        subjective_notes='No human subjective score yet.',
        metrics=summary,
        case_results=case_results,
    )

    for row in compact_rows(case_results):
        row['combo_id'] = combo_id
        all_result_rows.append(row)

all_result_rows[:10]


## 9. Compact post-run status

Keep notebook inspection light here. Detailed answer comparison should move to the future Streamlit review app.


In [ ]:
if 'experiment' not in globals():
    raise RuntimeError('Load or create an experiment first.')

experiment_state = garden.get_experiment(EXPERIMENT_ID)
{
    'experiment_id': experiment_state['id'],
    'status': experiment_state['status'],
    'attached_combo_count': len(experiment_state.get('combo_ids', [])),
    'tested_combo_count': experiment_state.get('summary', {}).get('tested_combo_count', 0),
    'average_score': experiment_state.get('summary', {}).get('average_score'),
    'best_score': experiment_state.get('summary', {}).get('best_score'),
    'worst_score': experiment_state.get('summary', {}).get('worst_score'),
}


In [ ]:
garden.experiment_results_table(EXPERIMENT_ID)


## 10. Finalize the experiment after human review

Keep finalization explicit so the notebook does not silently close experiments.


In [ ]:
FINALIZE_EXPERIMENT = False
FINAL_RESULT_TEXT = 'Summarize the outcome after script execution and review.'
FINAL_SUBJECT_SCORE = None

if FINALIZE_EXPERIMENT:
    garden.finalize_experiment(
        experiment_id=EXPERIMENT_ID,
        result_text=FINAL_RESULT_TEXT,
        subject_score=FINAL_SUBJECT_SCORE,
        status='completed',
    )
    garden.get_experiment(EXPERIMENT_ID)
else:
    print('Set FINALIZE_EXPERIMENT = True when you are ready to close the experiment.')
